In [0]:
# ==========================================
# Standalone Data Quality & Audit Notebook
# ==========================================

# 1. Read the Silver Cleaned Table
silver_df = spark.read.table("realtime_fraud_detection._02_silver.cleaned_transactions")

# 2. Read the Quarantine Table (to audit bad records caught by the stream)
quarantine_df = spark.read.table("realtime_fraud_detection._02_silver.quarantine_transactions")

print(f"📊 --- Silver Table Audit Summary ---")
print(f"Total Clean Records: {silver_df.count()}")

print(f"\n⚠️ --- Quarantine Table Audit Summary ---")
quarantine_count = quarantine_df.count()
print(f"Total Quarantined (Bad) Records: {quarantine_count}")

# If there are quarantined records, show a sample so you can investigate why they failed
if quarantine_count > 0:
    print("\n🔍 Sample of Quarantined Records:")
    display(quarantine_df.limit(10))

# 3. Final Assertions on the Clean Silver Table
invalid_amounts = silver_df.filter(silver_df.Amount < 0).count()
invalid_classes = silver_df.filter(~silver_df.Class.isin([0, 1])).count()
null_checks = silver_df.filter(
    silver_df.Time.isNull() | silver_df.Amount.isNull() | silver_df.Class.isNull()
).count()

print(f"\n--- Silver Table Data Quality Assertion Results ---")
print(f"❌ Negative/Invalid Amounts: {invalid_amounts}")
print(f"❌ Invalid Class Labels: {invalid_classes}")
print(f"❌ Null values in core columns: {null_checks}")

# Hard assertions to ensure the Silver table is pristine
assert invalid_amounts == 0, "Data Quality Error: Negative transaction amounts found in Silver!"
assert invalid_classes == 0, "Data Quality Error: Invalid classes found in Silver!"
assert null_checks == 0, "Data Quality Error: Null values found in Silver core columns!"

print("\n✅ All Silver Table Data Quality & Expectation Checks Passed Successfully!")